In [59]:
def pp(msg=None, value=None):
    if msg is not None:
        print(msg)
    if value is not None:
        print(value)
    print('=' * 100)

In [60]:
import pandas as pd

flights_raw = pd.read_csv('data/flights.csv')
flights = flights_raw.copy()
pp("flights.csv", flights.shape)

pp("Columns", flights.columns)
pp("Dtypes", flights.dtypes)

flights.csv
(20000, 5)
Columns
Index(['date', 'delay', 'distance', 'origin', 'destination'], dtype='str')
Dtypes
date             str
delay          int64
distance       int64
origin           str
destination      str
dtype: object


Every row is a single flight from origin to destination airport:

In [61]:
pp("Sample flight", flights.head(1))

Sample flight
                  date  delay  distance origin destination
0  2001-01-01 00:47:00     66      1750    DTW         LAS


# Data cleaning
Data problems:
1. `date` as string
2. `distance` unit not specified 
3. `delay` has negative values

In [62]:
# 1. `date` as string
# Convert `date` str to datetime using Pandas (20000 rows):
flights['date'] = pd.to_datetime(flights['date'])
flights.dtypes


date           datetime64[us]
delay                   int64
distance                int64
origin                    str
destination               str
dtype: object

In [63]:
# 2. `distance` unit not specified 
# Considering USA flight data, we can assume miles as distance unit.
# Let's make a quick check of distances for first 3 flights (https://www.greatcirclemap.com/):
# DTW -> LAS = 1749 miles
# HNL -> SFO = 2398 miles
# LAS -> OAK = 407 miles
# for distance in flights.head(3)['distance']:
#     print(distance)
# # 1750
# # 2399
# # 407
# As assumption is correct, let's rename the column:
flights.rename(columns={'distance': 'distance_miles'}, inplace=True)
flights.columns

Index(['date', 'delay', 'distance_miles', 'origin', 'destination'], dtype='str')

In [64]:
# 3. `delay` has negative values
# Negative/zero delay means early/no delay arrival.
# A separate boolean value can be useful for further evaluations (20000 rows):
flights['is_delayed'] = flights['delay'] > 0
pp("Total of flights", len(flights))
flights['is_delayed'].value_counts(normalize=True)

Total of flights
20000


is_delayed
False    0.52535
True     0.47465
Name: proportion, dtype: float64

In [65]:
# As the description says that `delay` is in minutes, let's adjust the corresponding column name:
flights.rename(columns={'delay': 'delay_minutes'}, inplace=True)
# And save cleaned data:
flights.to_csv('data/flights_clean.csv', index=False)
flights.dtypes

date              datetime64[us]
delay_minutes              int64
distance_miles             int64
origin                       str
destination                  str
is_delayed                  bool
dtype: object

# Missing values analysis

There is NO missing values in columns (i.e. we keep everything).
Though, `delay` and `distanse` column names have been renamed in previous steps to show units:
`delay` -> `delay_minutes`
and
`distance` -> `distance_miles`.
This is important for furter analysis, e.g. for `longer distance -> higher delay` correlation.
Without understanding the units of measure, it can be difficult to evaluate

In [66]:

print(flights.isna().sum())

date              0
delay_minutes     0
distance_miles    0
origin            0
destination       0
is_delayed        0
dtype: int64


# Merging tables (Flights and Airports data)
For further analysis flights.csv and airports.csv can be merged on `iata` of each airport.
As a result additional airport data will be available: name, city, state, country, latitude and longitude:

In [67]:
airports_raw = pd.read_csv("data/airports.csv")
airports = airports_raw.copy()
pp("flights.csv", flights.shape)
pp(airports.dtypes)

pp("Flights before merge", len(flights))
pp("Airports", len(airports))

origin_airports = airports.rename(columns={
    "iata": "origin",
    "name": "origin_airport_name",
    "city": "origin_city",
    "state": "origin_state",
    "country": "origin_country",
    "latitude": "origin_latitude",
    "longitude": "origin_longitude"
})

destination_airports = airports.rename(columns={
    "iata": "destination",
    "name": "destination_airport_name",
    "city": "destination_city",
    "state": "destination_state",
    "country": "destination_country",
    "latitude": "destination_latitude",
    "longitude": "destination_longitude"
})

merged = flights.merge(
    origin_airports,
    on="origin",
    how="left",
    validate="many_to_one"
)
pp("After origin merge", len(merged))

merged = merged.merge(
    destination_airports,
    on="destination",
    how="left",
    validate="many_to_one"
)
pp("After destination merge", len(merged))

pp("Missing origin airports", merged["origin_airport_name"].isna().sum())
pp("Missing destination airports", merged["destination_airport_name"].isna().sum())
pp("Duplicated rows", merged.duplicated().sum())

merged.to_csv('data/merged.csv', index=False)
pp("merged.csv", merged.shape)
pp(merged.dtypes)

flights.csv
(20000, 6)
iata             str
name             str
city             str
state            str
country          str
latitude     float64
longitude    float64
dtype: object
Flights before merge
20000
Airports
3376
After origin merge
20000
After destination merge
20000
Missing origin airports
0
Missing destination airports
0
Duplicated rows
0
merged.csv
(20000, 18)
date                        datetime64[us]
delay_minutes                        int64
distance_miles                       int64
origin                                 str
destination                            str
is_delayed                            bool
origin_airport_name                    str
origin_city                            str
origin_state                           str
origin_country                         str
origin_latitude                    float64
origin_longitude                   float64
destination_airport_name               str
destination_city                       str
destination_state   

# Adding new columns
We already added `is_delayed` column, second useful one can be `weekday` of a delay (e.g. it can be used for evaluation of most common days of flights delay): 

In [68]:
merged['weekday'] = pd.to_datetime(merged['date']).dt.day_name()

pp(merged['weekday'].value_counts())
# pp(merged['weekday'].value_counts(normalize=True).round(2))

merged.to_csv('data/merged.csv', index=False)
pp("merged.csv", merged.shape)
pp(merged.dtypes)

weekday
Friday       3003
Monday       2952
Thursday     2944
Tuesday      2903
Wednesday    2895
Saturday     2655
Sunday       2648
Name: count, dtype: int64
merged.csv
(20000, 19)
date                        datetime64[us]
delay_minutes                        int64
distance_miles                       int64
origin                                 str
destination                            str
is_delayed                            bool
origin_airport_name                    str
origin_city                            str
origin_state                           str
origin_country                         str
origin_latitude                    float64
origin_longitude                   float64
destination_airport_name               str
destination_city                       str
destination_state                      str
destination_country                    str
destination_latitude               float64
destination_longitude              float64
weekday                                str
